# Model C, C2 — diagnostic: old CL vs. v09, and committed time by phase in Full CC (Oct 31, 2026) — EXPLORATORY, descriptive

**Spec (as given):**

> TASK: C2 diagnostic, descriptive only, no model change. Never load Ring/CT.
> 1. Re-simulate C2 for 10 replicates (seed 8001 streams, L3/L5 design).
> 2. For Full CL and Hierarchy CL, split rounds into old-CL vs. v09 and report, for humans
>    and C2: number of rounds, success rate, KM median time to consensus, committed split %,
>    and the C2 counts of tries, kept runs and walk steps per round.
> 3. For Full CC: share of round time in committed splits by phase (first 10 s vs. after),
>    humans vs. C2.
> Report the tables. Commit. Stop.


## Choices

- **Replicates.** The C2 main run's first batch exactly: `design.simulate_rounds(d3, "C2", R=20, seed=[8001, 0, 0])` on the L3 design, as `p24_main.py` ran it (batch 0, seed [seed, batch, first replicate]). Only replicates 0-9 are used. R = 20 is needed because the 20 copies share one random stream, so R = 10 would give other draws. The check below compares targets 1, 2, 4 and 6 of these 10 replicates with `p24_reps_C2.parquet` (they must be equal).
- **Design.** Targets 1-4 use L3 rounds (`load_runs("outcome", roles={"train", "val"})`: G-train + G-val, whole-second old-CL files included), as in the checks. L5 is not needed: the asked statistics are all L3 targets. Ring and CT are not in L3.
- **Old CL vs. v09.** The design's `interface` (`interface.version_tag(file) == "v09"` → lab_v09, else old_cl); human rounds get the same label (same order as the design).
- **Statistics.** As `targets.check_stats`: success % = mean of success; KM median = `km_quantile(kaplan_meier(ttc, success), 0.5)`, failures censored at the time limit; committed % = mean per round of `h5.committed_spells`' % committed (cutoff 0.9). For C2 they are pooled over the 10 replicates' rounds: the same as the mean over replicates for success and committed %, and one KM curve for the median. NaN = the median is not reached (success < 50%); target 2 would then use the time limit.
- **Tries, kept runs, walk steps (C2 only).** Counted by a wrapper around `sim.tick_c2` that reads the state before each tick and the step after it, and the look frames of the simulator's own trace (the other trace frames are dropped). The simulation is unchanged: the wrapper draws nothing and changes no state (checked bit-identical on a few rounds, and by the replicate check below). Try = a step by a player sitting at its wall; kept run = a look that keeps; walk step = a step by a walking player (after a look that did not keep). Per round = summed over the 7 seats. Looks are counted too (looks = tries + walk steps, minus the walk steps that reach the wall and the steps whose look is cut off by the round end).
- **Phases (Full CC).** For each round, the committed time (forward hold, as in `h5.committed_spells`) is split at t = 10 s: [0, 10) and [10, t_end). Two numbers per phase: the pooled share of the phase's round time in committed splits (sum over rounds of committed time / sum of time in the phase), and the phase's part of target 4 (mean over rounds of committed time in the phase / t_end; the two parts add up to target 4).


In [1]:
import warnings

import numpy as np
import pandas as pd

from slider import config, design, h5, sim, targets
from slider import slider_logs as sl
from slider.fraught import committed_mask, round_end
from slider.stats import kaplan_meier

warnings.filterwarnings("ignore", message="Pandas requires version")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

SEED, R_BATCH, R = [8001, 0, 0], 20, 10
d3 = design.design("L3")
runs_h, table_h = design.load("L3")
assert set(d3["topology"]) == {"Full", "Hierarchy", "Star"}               # no Ring, no CT
assert list(table_h["file"]) == list(d3["group"]) and list(table_h["wave"]) == list(d3["wave"])
d3.groupby(["topology", "dataset", "interface"]).size().rename("rounds").to_frame().T


topology  Full                Hierarchy                        Star
dataset     CC      CL               CC      CL             EL   CC
interface   cc lab_v09 old_cl        cc lab_v09 old_cl lab_v09   cc
rounds      36      39     24        40      39     22      51   39

**How to read it.** The L3 design: rounds per cell and interface. Full CL and Hierarchy CL mix v09 and old-CL rounds; each C2 replicate simulates every one of them once.

In [2]:
ORIG = sim.tick_c2
TALLY = []      # one dict per sim.simulate call (one round of the design, in order): counts per replicate


class Looks(list):
    """A simulator trace that keeps only the frames of the C2 looks."""
    def append(self, df):
        if "look" in df.columns:
            super().append(df)


def counting_tick(st, j, c, rng):
    """sim.tick_c2, plus counts of tries, looks, kept runs and walk steps per replicate (reads state only)."""
    if j == 0:                                                   # a new sim.simulate call
        TALLY.append({k: np.zeros(len(st["id"]), dtype=int) for k in ["tries", "looks", "kept", "walk"]})
        c["trace"] = Looks()
    t = TALLY[-1]
    ids = st["id"].copy()
    sit = st["mode"] == sim.AT_WALL
    walking = st["c2_walk"].copy()
    decide = st["c2_wait"] == j
    during, step = ORIG(st, j, c, rng)
    kept = np.zeros_like(decide)
    for df in c["trace"]:
        kept[np.searchsorted(ids, df["rep"].to_numpy()), df["player"].to_numpy()] = df["kept"].to_numpy()
    c["trace"].clear()
    moved = step != 0
    t["tries"][ids] += (sit & moved).sum(axis=1)
    t["looks"][ids] += decide.sum(axis=1)
    t["kept"][ids] += kept.sum(axis=1)
    t["walk"][ids] += ((walking | (decide & ~kept)) & moved).sum(axis=1)
    return during, step


sim.tick_c2 = counting_tick
try:
    rounds = design.simulate_rounds(d3, "C2", R=R_BATCH, seed=SEED)
finally:
    sim.tick_c2 = ORIG
assert len(TALLY) == len(d3)


In [3]:
def committed_phases(runs, table, t0=10.0):
    """Per run: committed time and round time before and after t0 (forward hold, cutoff 0.9, as h5)."""
    rows = []
    for k, run in enumerate(runs):
        t = run["t"]
        mask = committed_mask(sl.sign_correct(run), h5.MAIN_CUTOFF)
        t_end = round_end(t, table["end"][k], table["session"][k])
        t1 = np.append(t[1:], t_end)
        early = np.clip(np.minimum(t1, t0) - t, 0, None)
        rows.append({"committed_early": np.sum(early * mask), "committed_late": np.sum((t1 - t - early) * mask),
                     "time_early": min(t_end, t0), "time_late": max(t_end - t0, 0.0), "t_end": t_end})
    return pd.DataFrame(rows)


def per_round(runs, table):
    """targets.per_run_values (success, ttc, committed %, end sitting) plus the committed phases."""
    r, _ = targets.per_run_values(runs, table)
    return pd.concat([r.reset_index(drop=True), committed_phases(runs, table)], axis=1)


roles = list(d3["role"])
sim_rounds = []
for r in range(R_BATCH):
    runs3 = design.copy_runs(d3, rounds, r, SEED)               # the logger streams are used in copy order
    if r < R:
        x = per_round(runs3, sim.table(runs3, roles))
        x["interface"], x["replicate"] = d3["interface"].to_numpy(), r
        for name in ["tries", "looks", "kept", "walk"]:
            x[name] = [t[name][r] for t in TALLY]
        sim_rounds.append(x)
sim_rounds = pd.concat(sim_rounds, ignore_index=True)
human = per_round(runs_h, table_h).assign(interface=d3["interface"].to_numpy())
del rounds


In [4]:
def cell_stats(r):
    """Targets 1, 2, 4, 6 of a set of rounds, as targets.check_stats."""
    km = targets.km_quantile(kaplan_meier(r["ttc"], r["success"])[0], 0.5)
    return pd.Series({1: 100 * r["success"].mean(), 2: targets.median_or_limit(km, r["session"]),
                      4: r["committed %"].mean(), 6: 100 * r["end time"].sum() / r["player time"].sum()})


saved = pd.read_parquet(config.P24_REPS["C2"])
saved = saved[(saved["replicate"] < R) & saved["target"].isin([1, 2, 4, 6]) & (saved["group"] == "")]
cols = ["success", "ttc", "session", "committed %", "end time", "player time"]
mine = (sim_rounds.groupby(["replicate", "topology", "dataset"])[cols].apply(cell_stats)
        .stack().rename("value").rename_axis(["replicate", "topology", "dataset", "target"]).reset_index())
both = mine.merge(saved, on=["replicate", "topology", "dataset", "target"], suffixes=("", "_saved"), validate="1:1")
print(f"{len(both)} values (10 replicates x 6 cells x 4 targets), max |difference| from p24_reps_C2.parquet: "
      f"{(both['value'] - both['value_saved']).abs().max():.2e}")
assert len(both) == R * 6 * 4 and np.allclose(both["value"], both["value_saved"], rtol=0, atol=1e-9)


240 values (10 replicates x 6 cells x 4 targets), max |difference| from p24_reps_C2.parquet: 0.00e+00


**How to read it.** These 10 replicates are the main run's replicates 0-9: their targets 1, 2, 4 and 6 equal the saved ones, so the counting wrapper did not change the simulation.

## Part 2 — Full CL and Hierarchy CL: old CL vs. v09

In [5]:
def split_stats(r, sim_counts):
    km = targets.km_quantile(kaplan_meier(r["ttc"], r["success"])[0], 0.5)
    out = {"rounds": len(r), "success %": 100 * r["success"].mean(), "KM median s": km,
           "committed %": r["committed %"].mean()}
    if sim_counts:
        out.update({f"{name} / round": r[name].mean() for name in ["tries", "kept", "walk", "looks"]})
    return pd.Series(out)


cl = []
for who, df in [("human", human), ("C2", sim_rounds)]:
    d = df[df["dataset"] == "CL"]
    for (topology, iface), r in d.groupby(["topology", "interface"]):
        cl.append(split_stats(r, who == "C2").rename((topology, iface, who)))
    for topology, r in d.groupby("topology"):
        cl.append(split_stats(r, who == "C2").rename((topology, "all CL", who)))
cl = pd.DataFrame(cl)
cl.index = pd.MultiIndex.from_tuples(cl.index, names=["topology", "interface", "who"])
order = [(tp, i, w) for tp in ["Full", "Hierarchy"] for i in ["old_cl", "lab_v09", "all CL"] for w in ["human", "C2"]]
cl = cl.loc[order]
cl["rounds"] = cl["rounds"].astype(int)
cl.round(2)


rounds  success %  KM median s  committed %  tries / round  kept / round  walk / round  looks / round
topology  interface who                                                                                                         
Full      old_cl    human      24      70.83       101.00         8.83            NaN           NaN           NaN            NaN
                    C2        240       7.92          NaN         0.94         111.66         87.98         60.68         148.68
          lab_v09   human      39      89.74        72.31         5.06            NaN           NaN           NaN            NaN
                    C2        390      80.77       153.30        11.42         129.06        100.78         73.51         174.32
          all CL    human      63      82.54        89.19         6.50            NaN           NaN           NaN            NaN
                    C2        630      53.02       260.20         7.43         122.43         95.90         68.62         164.55
Hierarchy old_cl    human      22      90.91        67.74         6.12            NaN           NaN           NaN            NaN
                    C2        220      27.27          NaN         1.79          76.39         59.72         44.09         103.82
          lab_v09   human      39      97.44        36.84         9.60            NaN           NaN           NaN            NaN
                    C2        390      97.95        59.10        14.08          44.95         34.97         25.86          60.83
          all CL    human      61      95.08        47.06         8.34            NaN           NaN           NaN            NaN
                    C2        610      72.46       111.00         9.65          56.29         43.90         32.43          76.33

**How to read it.** Per topology and interface, humans (one round per L3 round) vs. C2 (10 replicates, so 10 x as many rounds). "all CL" is the whole cell, as in the checks: its success and committed % match the gap table's C2 values only up to the replicates used here (10 vs. 200). KM median NaN = not reached (success < 50%). The counts are means per round, summed over the 7 seats: tries (steps off a wall), kept runs, walk steps and looks.

## Part 3 — Full CC: committed time by phase

In [6]:
def phase_stats(r):
    out = {"rounds": len(r), "rounds past 10 s": int((r["t_end"] > 10).sum()), "committed % (target 4)": r["committed %"].mean()}
    for phase in ["early", "late"]:
        out[f"share {phase} %"] = 100 * r[f"committed_{phase}"].sum() / r[f"time_{phase}"].sum()
        out[f"part of target 4, {phase}"] = (100 * r[f"committed_{phase}"] / r["t_end"]).mean()
    out["mean round s"] = r["t_end"].mean()
    out["round time past 10 s, share %"] = 100 * r["time_late"].sum() / r["t_end"].sum()
    return pd.Series(out)


fcc = pd.DataFrame({who: phase_stats(df[(df["topology"] == "Full") & (df["dataset"] == "CC")])
                    for who, df in [("human", human), ("C2", sim_rounds)]}).T
assert np.allclose(fcc["part of target 4, early"] + fcc["part of target 4, late"], fcc["committed % (target 4)"])
fcc.round(2)


,rounds,rounds past 10 s,committed % (target 4),share early %,"part of target 4, early",share late %,"part of target 4, late",mean round s,"round time past 10 s, share %"
human,36.0,36.0,14.61,1.89,0.72,20.82,13.89,57.44,82.59
C2,360.0,357.0,30.19,0.67,0.11,39.68,30.08,93.03,89.26


**How to read it.** Full CC, humans vs. C2 (10 replicates). Early = the first 10 s of a round, late = the rest. `share * %` = the share of all round time in that phase spent in committed splits (pooled over rounds). `part of target 4, *` = the mean over rounds of (committed time in the phase / round length); the two parts add up to target 4. The last two columns give the round lengths, which set how much the late phase weighs.

**What the tables show (descriptive only; C2 is EXPLORATORY and decides nothing).**
- **CL: the miss is mostly old CL.** Old-CL rounds: C2 success 7.9% vs. human 70.8% (Full) and 27.3% vs. 90.9% (Hierarchy), no KM median, and almost no committed time (0.9% vs. 8.8%, 1.8% vs. 6.1%), even though C2 makes as many or more tries per round there than on v09. On v09, success is close (Full 80.8% vs. 89.7%, Hierarchy 98.0% vs. 97.4%), but C2 is still slower (KM median 153 vs. 72 s, 59 vs. 37 s) and spends more time committed (11.4% vs. 5.1%, 14.1% vs. 9.6%).
- **Full CC: the extra committed time is after the first 10 s.** In the first 10 s both are near 0 (human 1.9%, C2 0.7%). After it, C2 spends 39.7% of the time in committed splits vs. 20.8% for humans, and its rounds are longer (mean 93 vs. 57 s), so the late phase weighs more: it gives 30.1 of C2's 30.2 points of target 4 (humans 13.9 of 14.6).
